# Keyword based curriculum analysis

## Keywords


In [ ]:
import regex as re
import pandas as pd
from pathlib import Path
from dataclasses import dataclass, field
from enum import Enum, auto

DEST_DIR = Path("keyword_results")
SENSITIVITY_DIR = Path("sensitivity")


# ============================================================
# ANKER-KEYWORDS (definieren das Kontextfenster)
# ============================================================

ANCHOR_PATTERNS = {
    "PW": re.compile(r"(Programmierwerkzeug(en?)?){e<=2}", re.IGNORECASE),
    "PS": re.compile(r"(Programmiersprache(en?)?){e<=2}", re.IGNORECASE),
    "PE": re.compile(r"(Programmierumgebung(en?)?){e<=2}", re.IGNORECASE),
    "EE": re.compile(r"(Entwicklungsumgebung(en?)?){e<=2}", re.IGNORECASE),
}

# ============================================================
# GLOBALE KEYWORDS: Konkrete Tools/Sprachen
# Werden im gesamten Dokument gesucht
# ============================================================

GLOBAL_KEYWORDS = {
    # --- Blockbasierte Tools ---
    "Scratch": re.compile(r"\bScratch\b", re.IGNORECASE),
    "Snap!": re.compile(r"\bSnap!?\b", re.IGNORECASE),
    "Blockly": re.compile(r"\bBlockly\b", re.IGNORECASE),
    "mBlock": re.compile(r"\bmBlock\b", re.IGNORECASE),
    "Open Roberta": re.compile(r"\bOpen\s*Roberta\b", re.IGNORECASE),
    "MakeCode": re.compile(r"\bMakeCode\b", re.IGNORECASE),
    "NEPO": re.compile(r"\bNEPO\b", re.IGNORECASE),
    "App Inventor": re.compile(r"\bApp\s*Inventor\b", re.IGNORECASE),
    # --- Textbasierte Sprachen ---
    "Python": re.compile(r"\bPython\b", re.IGNORECASE),
    "Java": re.compile(r"\bJava\b(?!\s*Script)", re.IGNORECASE),
    "JavaScript": re.compile(r"\bJava\s*Script\b", re.IGNORECASE),
    "C#": re.compile(r"\bC\s*#\b", re.IGNORECASE),
    "C++": re.compile(r"\bC\s*\+\+\b", re.IGNORECASE),
    # "C": re.compile(r"\bC\b"),
    "Delphi": re.compile(r"\bDelphi\b", re.IGNORECASE),
    "Pascal": re.compile(r"\bPascal\b", re.IGNORECASE),
    "Haskell": re.compile(r"\bHaskell\b", re.IGNORECASE),
    "Perl": re.compile(r"\bPerl\b", re.IGNORECASE),
    "Prolog": re.compile(r"\bProlog\b", re.IGNORECASE),
    "PHP": re.compile(r"\bPHP\b", re.IGNORECASE),
    "Lisp": re.compile(r"\bLisp\b", re.IGNORECASE),
    "Scheme": re.compile(r"\bScheme\b", re.IGNORECASE),
    "Visual Basic": re.compile(r"\bVisual\s*Basic\b", re.IGNORECASE),
    # --- Educational Languages / Minisprachen ---
    "Robot Karol": re.compile(r"\bRobot\s*Karol\b", re.IGNORECASE),
    "Kara": re.compile(r"\bKara\b", re.IGNORECASE),
    "Logo": re.compile(r"\bLogo\b", re.IGNORECASE),
    "Hamster": re.compile(r"\bHamster-?w*\b", re.IGNORECASE),
    # --- IDEs / Umgebungen ---
    "Greenfoot": re.compile(r"\bGreenfoot\b", re.IGNORECASE),
    "BlueJ": re.compile(r"\bBlueJ\b", re.IGNORECASE),
    "EOS": re.compile(r"\bEOS\b"),
    "Thonny": re.compile(r"\bThonny\b", re.IGNORECASE),
    "Eclipse": re.compile(r"\bEclipse\b", re.IGNORECASE),
    "IntelliJ": re.compile(r"\bIntelliJ\b", re.IGNORECASE),
    "VS Code": re.compile(r"\b(VS\s*Code|Visual\s*Studio)\b", re.IGNORECASE),
    "Jupyter": re.compile(r"\bJupyter\b", re.IGNORECASE),
    # --- Physical Computing Hardware ---
    "Arduino": re.compile(r"\bArduino\b", re.IGNORECASE),
    "Calliope": re.compile(r"\bCalliope\b", re.IGNORECASE),
    "micro:bit": re.compile(r"\bmicro\s*[:\-]?\s*bit\b", re.IGNORECASE),
    "Raspberry Pi": re.compile(r"\bRaspberry\s*Pi\b", re.IGNORECASE),
    "Lego Mindstorms": re.compile(r"\bLego\s*Mindstorms\b", re.IGNORECASE),
    "Lego Spike": re.compile(r"\bLego\s*Spike\b", re.IGNORECASE),
    "Fischer Technik": re.compile(r"\bFischer\s*Technik\b", re.IGNORECASE),
    "BOB3": re.compile(r"\bBOB\s*3\b", re.IGNORECASE),
    "Ozobot": re.compile(r"\bOzobot\b", re.IGNORECASE),
    "Bee-Bot": re.compile(r"\bBee[\s\-]?Bot\b", re.IGNORECASE),
    "Blue-Bot": re.compile(r"\bBlue[\s\-]?Bot\b", re.IGNORECASE),
    # Misc
    "Scriptsprache": re.compile(
        r"\b(S(k|c)ript(ing)?sprachen?){e<=2}\b", re.IGNORECASE
    ),
}

# ============================================================
# KONTEXT-KEYWORDS: Nur im Fenster um Anker gesucht
# ============================================================

CONTEXT_KEYWORDS = {
    # --- Interaktionsparadigma ---
    "ctx_blockbasiert": re.compile(r"\bblock[\s\-]?basiert\w*\b", re.IGNORECASE),
    "ctx_textbasiert": re.compile(r"\btext([\s\-]?basiert|uell)\w*\b", re.IGNORECASE),
    "ctx_visuell": re.compile(r"\bvisuel\w+\b", re.IGNORECASE),
    "ctx_grafisch": re.compile(r"\bgrafisch\w*\b", re.IGNORECASE),
    "ctx_hybrid": re.compile(r"\bhybrid\w*\b", re.IGNORECASE),
    # --- Programmierparadigma ---
    "ctx_objektorientiert": re.compile(
        r"\bobjekt[\s\-]?orientiert\w*\b", re.IGNORECASE
    ),
    "ctx_funktional": re.compile(r"\bfunktional\w*\b", re.IGNORECASE),
    "ctx_imperativ": re.compile(r"\bimperativ\w*\b", re.IGNORECASE),
    "ctx_prozedural": re.compile(r"\bprozedural\w*\b", re.IGNORECASE),
    "ctx_deklarativ": re.compile(r"\bdeklarativ\w*\b", re.IGNORECASE),
    "ctx_ereignisgesteuert": re.compile(
        r"\bereignis(?:gesteuert|orientiert)\w*\b", re.IGNORECASE
    ),
    "ctx_event": re.compile(
        r"\bevent[\s\-]?(?:driven|basiert|orientiert)\w*\b", re.IGNORECASE
    ),
    # --- Verbindlichkeitsgrad ---
    # "ctx_verpflichtend": re.compile(r"\bverpflichtend\w*\b", re.IGNORECASE),
    # "ctx_verbindlich": re.compile(r"\bverbindlich\w*\b", re.IGNORECASE),
    # "ctx_ist_zu_verwenden": re.compile(r"\bist\s+zu\s+verwenden\b", re.IGNORECASE),
    # "ctx_wird_verwendet": re.compile(r"\bwird\s+verwendet\b", re.IGNORECASE),
    # "ctx_empfohlen": re.compile(r"\bempfohlen\b", re.IGNORECASE),
    # "ctx_empfiehlt_sich": re.compile(r"\bempfiehlt\s+sich\b", re.IGNORECASE),
    # "ctx_sollte": re.compile(r"\bsollte\w*\b", re.IGNORECASE),
    # "ctx_bietet_sich_an": re.compile(r"\bbietet\s+sich\s+an\b", re.IGNORECASE),
    # "ctx_beispielsweise": re.compile(r"\bbeispielsweise\b", re.IGNORECASE),
    # "ctx_z_b": re.compile(r"\bz\.\s*B\.\b"),
    # "ctx_wie_etwa": re.compile(r"\bwie\s+etwa\b", re.IGNORECASE),
    # "ctx_unter_anderem": re.compile(r"\bunter\s+anderem\b", re.IGNORECASE),
    # "ctx_geeignet": re.compile(r"\bgeeignet\w*\b", re.IGNORECASE),
    # "ctx_kann": re.compile(r"\bkann\b", re.IGNORECASE),
    # "ctx_koennen": re.compile(r"\bkönnen\b", re.IGNORECASE),
    # --- Werkzeugeigenschaften ---
    "ctx_pädagogisch": re.compile(r"\bpädagogisch\w*\b", re.IGNORECASE),
    "ctx_didaktisch": re.compile(r"\bdidaktisch\w*\b", re.IGNORECASE),
    "ctx_altersgerecht": re.compile(r"\balters(gerecht|angemessen)\w*\b", re.IGNORECASE),
    "ctx_schülergerecht": re.compile(r"\bschülergerecht\w*\b", re.IGNORECASE),
    "ctx_kindgerecht": re.compile(r"\bkindgerecht\w*\b", re.IGNORECASE),
    "ctx_einfach": re.compile(r"\beinfach\w*\b", re.IGNORECASE),
    "ctx_komplex": re.compile(r"\bkomplex\w*\b", re.IGNORECASE),
    "ctx_professionell": re.compile(r"\bprofessionell\w*\b", re.IGNORECASE),
    "ctx_industrienah": re.compile(r"\bindustrienah\w*\b", re.IGNORECASE),
    "ctx_aktuell": re.compile(r"\baktuell\w*\b", re.IGNORECASE),
    "ctx_modern": re.compile(r"\bmodern\w*\b", re.IGNORECASE),
    # --- Physical Computing Kontext ---
    "ctx_physical_computing": re.compile(r"\bPhysical\s*Computing\b", re.IGNORECASE),
    "ctx_mikrocontroller": re.compile(r"\bMikrocontroller\w*\b", re.IGNORECASE),
    "ctx_roboter": re.compile(r"\bRobot(er|ik)\w*\b", re.IGNORECASE),
}

## Settings and Datattypes for Analysis


In [ ]:
SOURCE_DIR = Path("../01_text_extraction/txt1")

PAGE_MARKER = re.compile(r"\[\[SEITE\s+(\d+)\]\]")


@dataclass
class Page:
    number: int
    text: str


class WindowMode(Enum):
    FULL = auto()
    PAGES = auto()  # ±n Seiten
    CHARS = auto()  # ±n Zeichen


WINDOW_MODE = WindowMode.PAGES
WINDOW_SIZE = 2


@dataclass
class Match:
    keyword_name: str
    matched_text: str
    fuzzy_counts: tuple  # (insertions, deletions, substitutions)
    total_errors: int
    page: int
    context: str
    search_type: str  # "global", "context", "anchor"

    def to_dict(self, filepath: Path) -> dict:
        """Converts Match to dictionary for dataframes

        Args:
            match (Match): Match object
            filepath (Path): Path to file of match

        Returns:
            dict: Match object as dictionary
        """
        # Ordnerstruktur: ../01_text_extraction/txt1/{Lehrpläne|GI}/{Unterordner}/datei.txt
        parts = filepath.relative_to(SOURCE_DIR).parts
        category = parts[0] if len(parts) > 0 else "unknown"  # "Lehrpläne" oder "GI"
        folder = (
            parts[1] if len(parts) > 1 else category
        )  # Bundesland oder GI-Subfolder

        # Bei GI: folder = "GI" (nicht den Unterordner)
        if category in ["GI", "KMK"]:
            folder = category

        return {
            "category": category,
            "folder": folder,
            "filename": filepath.name,
            "filepath": str(filepath),
            "keyword_name": self.keyword_name,
            "matched_text": self.matched_text,
            "fuzzy_counts": self.fuzzy_counts,
            "total_errors": self.total_errors,
            "page": self.page,
            "context": self.context,
            "search_type": self.search_type,
        }


@dataclass
class AnchorHit:
    """Ein Anker-Treffer mit Position im Gesamttext."""

    name: str
    match: Match
    char_start: int  # Position im Gesamttext
    char_end: int  # Position im Gesamttext
    page_index: int  # Index in pages-Liste

## Functions

### Page parsing


In [ ]:
def parse_pages(text: str) -> list[Page]:
    markers = list(PAGE_MARKER.finditer(text))
    if not markers:
        return [Page(number=1, text=text)]

    pages = []
    if markers[0].start() > 0:
        pages.append(Page(number=0, text=text[: markers[0].start()].strip()))

    for i, marker in enumerate(markers):
        page_num = int(marker.group(1))
        start = marker.end()
        end = markers[i + 1].start() if i + 1 < len(markers) else len(text)
        page_text = text[start:end].strip()
        if page_text:
            pages.append(Page(number=page_num, text=page_text))

    return pages

### Keyword search


In [ ]:
def search_keyword(
    text: str,
    name: str,
    pattern: re.Pattern,
    page_number: int = -1,
    search_type: str = "global",
    context_chars: int = 80,
) -> list[Match]:
    matches = []
    for m in pattern.finditer(text):
        fuzzy = getattr(m, "fuzzy_counts", (0, 0, 0))
        total_err = sum(fuzzy)

        matched_text = m.group()
        match_start = m.start()
        match_end = m.end()

        # Bei Fuzzy-Treffern: versuche exakten, kürzeren Match zu finden
        if total_err > 0:
            # Exaktes Pattern aus dem ursprünglichen Suchbegriff
            # innerhalb des fuzzy-gematchten Bereichs + kleinem Puffer suchen
            search_start = max(0, match_start - total_err)
            search_end = min(len(text), match_end + total_err)
            search_region = text[search_start:search_end]

            # Exakte Suche (gleiche Flags ohne BESTMATCH, ohne fuzzy)
            exact_pat = re.compile(
                re.sub(
                    r"\{[^}]*[edis]<=[^}]*\}", "", pattern.pattern
                ),  # fuzzy-Suffix entfernen
                pattern.flags & ~re.BESTMATCH,
            )
            exact_m = exact_pat.search(search_region)

            if exact_m:
                # Exakten Match übernehmen
                matched_text = exact_m.group()
                match_start = search_start + exact_m.start()
                match_end = search_start + exact_m.end()
                fuzzy = (0, 0, 0)
                total_err = 0

        ctx_start = max(0, match_start - context_chars)
        ctx_end = min(len(text), match_end + context_chars)

        matches.append(
            Match(
                keyword_name=name,
                matched_text=matched_text,
                fuzzy_counts=fuzzy,
                total_errors=total_err,
                page=page_number,
                context=text[ctx_start:ctx_end].replace("\n", " "),
                search_type=search_type,
            )
        )
    return matches


def find_anchors_with_positions(
    pages: list[Page],
) -> list[AnchorHit]:
    """Findet alle Anker-Treffer mit ihren Positionen im Gesamttext."""
    anchors = []
    char_offset = 0

    for i, page in enumerate(pages):
        for name, pattern in ANCHOR_PATTERNS.items():
            for m in search_keyword(
                text=page.text,
                name=name,
                pattern=pattern,
                page_number=page.number,
                search_type="anchor",
            ):
                # Position des matched_text im Seitentext finden
                local_pos = page.text.find(m.matched_text)
                if local_pos == -1:
                    local_pos = 0

                anchors.append(
                    AnchorHit(
                        name=name,
                        match=m,
                        char_start=char_offset + local_pos,
                        char_end=char_offset + local_pos + len(m.matched_text),
                        page_index=i,
                    )
                )
        # +1 für den \n zwischen Seiten
        char_offset += len(page.text) + 1

    return anchors

### Context windows


In [ ]:
def build_context_windows_chars(
    full_text: str,
    anchors: list[AnchorHit],
    window_size: int = 500,
) -> list[tuple[str, list[int]]]:
    """Zeichenbasierte Kontextfenster (±n Zeichen). Überlappende werden gemergt."""
    if not anchors:
        return []

    # Rohe Intervalle sammeln
    intervals = []
    for anchor in anchors:
        start = max(0, anchor.char_start - window_size)
        end = min(len(full_text), anchor.char_end + window_size)
        intervals.append((start, end, anchor.match.page))

    # Nach Startposition sortieren
    intervals.sort(key=lambda x: x[0])

    # Überlappende Intervalle mergen
    merged = [intervals[0]]
    for start, end, page in intervals[1:]:
        prev_start, prev_end, prev_page = merged[-1]
        if start <= prev_end:
            merged[-1] = (prev_start, max(prev_end, end), prev_page)
        else:
            merged.append((start, end, page))

    windows = []
    for start, end, page in merged:
        window_text = full_text[start:end]
        windows.append((window_text, [page]))

    return windows


def build_context_windows_pages(
    pages: list[Page],
    anchors: list[AnchorHit],
    window_size: int = 2,
) -> list[tuple[str, list[int]]]:
    """Seitenbasierte Kontextfenster (±n Seiten). Überlappende werden gemergt."""
    if not anchors:
        return []

    context_indices = set()
    for anchor in anchors:
        for offset in range(-window_size, window_size + 1):
            target = anchor.page_index + offset
            if 0 <= target < len(pages):
                context_indices.add(target)

    sorted_indices = sorted(context_indices)
    ranges = []
    current_range = [sorted_indices[0]]
    for idx in sorted_indices[1:]:
        if idx == current_range[-1] + 1:
            current_range.append(idx)
        else:
            ranges.append(current_range)
            current_range = [idx]
    ranges.append(current_range)

    windows = []
    for r in ranges:
        combined_text = "\n".join(pages[i].text for i in r)
        page_numbers = [pages[i].number for i in r]
        windows.append((combined_text, page_numbers))

    return windows


def build_context_windows(
    pages: list[Page],
    full_text: str,
    anchors: list[AnchorHit],
    mode: WindowMode = WindowMode.PAGES,
    window_size: int = 2,
) -> list[tuple[str, list[int]]]:
    """Dispatcher: wählt seitenbasiert oder zeichenbasiert."""
    if mode == WindowMode.PAGES:
        return build_context_windows_pages(pages, anchors, window_size)
    elif mode == WindowMode.CHARS:
        return build_context_windows_chars(full_text, anchors, window_size)
    elif mode == WindowMode.FULL:
        return [(full_text, [-1])]
    else:
        raise ValueError(f"Unknown WindowMode: {mode}")

### Analysis


In [ ]:
def analyze_document(filepath: Path) -> list[dict]:
    text = filepath.read_text(encoding="utf-8")
    pages = parse_pages(text)
    full_text = "\n".join(p.text for p in pages)

    results = []

    # --- 1. Globale Suche ---
    for name, pattern in GLOBAL_KEYWORDS.items():
        for page in pages:
            for match in search_keyword(
                text=page.text,
                name=name,
                pattern=pattern,
                page_number=page.number,
                search_type="global",
            ):
                results.append(match.to_dict(filepath))

    # --- 2. Anker finden (mit Positionen) ---
    anchors = find_anchors_with_positions(pages)

    # Anker-Matches erfassen
    for anchor in anchors:
        results.append(anchor.match.to_dict(filepath))

    # --- 3. Kontextfenster bauen und durchsuchen ---
    windows = build_context_windows(
        pages=pages,
        full_text=full_text,
        anchors=anchors,
        mode=WINDOW_MODE,
        window_size=WINDOW_SIZE,
    )

    for window_text, page_numbers in windows:
        for name, pattern in CONTEXT_KEYWORDS.items():
            for match in search_keyword(
                text=window_text,
                name=name,
                pattern=pattern,
                search_type="context",
            ):
                match.page = page_numbers[0]
                results.append(match.to_dict(filepath))

    return results


def analyze_corpus() -> pd.DataFrame:
    all_results = []
    txt_files = list(SOURCE_DIR.rglob("*.txt"))
    print(f"Gefunden: {len(txt_files)} Dokumente")

    for i, filepath in enumerate(sorted(txt_files)):
        print(f"  [{i+1}/{len(txt_files)}] {filepath.relative_to(SOURCE_DIR)}")
        try:
            doc_results = analyze_document(filepath)
            all_results.extend(doc_results)
        except Exception as e:
            print(f"    FEHLER: {e}")

    df = pd.DataFrame(all_results)
    print(f"\nGesamt: {len(df)} Treffer")
    return df

### Aggregation


In [ ]:
def aggregate_by_document(df: pd.DataFrame) -> pd.DataFrame:
    """
    Pivot-Tabelle: eine Zeile pro Dokument,
    eine Spalte pro Keyword mit Anzahl der Treffer.
    """
    if df.empty:
        return pd.DataFrame()

    # Anzahl Treffer pro Dokument × Keyword
    doc_kw = (
        df.groupby(["category", "folder", "filename", "keyword_name"])
        .size()
        .reset_index(name="count")
    )

    # Pivotieren
    doc_pivot = doc_kw.pivot_table(
        index=["category", "folder", "filename"],
        columns="keyword_name",
        values="count",
        fill_value=0,
        aggfunc="sum",
    )

    # Zusätzliche Aggregatspalten
    global_kw_names = GLOBAL_KEYWORDS.keys()
    context_kw_names = CONTEXT_KEYWORDS.keys()
    anchor_kw_names = ANCHOR_PATTERNS.keys()

    present_global = [c for c in doc_pivot.columns if c in global_kw_names]
    present_context = [c for c in doc_pivot.columns if c in context_kw_names]
    present_anchor = [c for c in doc_pivot.columns if c in anchor_kw_names]

    doc_pivot["_total_global"] = (
        doc_pivot[present_global].sum(axis=1) if present_global else 0
    )
    doc_pivot["_total_context"] = (
        doc_pivot[present_context].sum(axis=1) if present_context else 0
    )
    doc_pivot["_total_anchor"] = (
        doc_pivot[present_anchor].sum(axis=1) if present_anchor else 0
    )
    doc_pivot["_total_all"] = (
        doc_pivot["_total_global"]
        + doc_pivot["_total_context"]
        + doc_pivot["_total_anchor"]
    )

    # Spalten sortieren: Summen zuerst, dann alphabetisch
    sum_cols = [c for c in doc_pivot.columns if c.startswith("_")]
    kw_cols = sorted([c for c in doc_pivot.columns if not c.startswith("_")])
    doc_pivot = doc_pivot[sum_cols + kw_cols]

    return doc_pivot.reset_index()


def aggregate_by_folder(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregiert auf Ordnerebene:
    - Lehrpläne: pro Bundesland-Unterordner
    - GI: zusammengefasst als "GI"
    """
    if df.empty:
        return pd.DataFrame()

    folder_kw = (
        df.groupby(["category", "folder", "keyword_name"])
        .size()
        .reset_index(name="count")
    )

    folder_pivot = folder_kw.pivot_table(
        index=["category", "folder"],
        columns="keyword_name",
        values="count",
        fill_value=0,
        aggfunc="sum",
    )

    # Anzahl Dokumente pro Ordner
    doc_counts = (
        df.groupby(["category", "folder"])["filename"]
        .nunique()
        .reset_index(name="_n_documents")
    )

    global_kw_names = GLOBAL_KEYWORDS.keys()
    context_kw_names = CONTEXT_KEYWORDS.keys()
    anchor_kw_names = ANCHOR_PATTERNS.keys()

    present_global = [c for c in folder_pivot.columns if c in global_kw_names]
    present_context = [c for c in folder_pivot.columns if c in context_kw_names]
    present_anchor = [c for c in folder_pivot.columns if c in anchor_kw_names]

    folder_pivot["_total_global"] = (
        folder_pivot[present_global].sum(axis=1) if present_global else 0
    )
    folder_pivot["_total_context"] = (
        folder_pivot[present_context].sum(axis=1) if present_context else 0
    )
    folder_pivot["_total_anchor"] = (
        folder_pivot[present_anchor].sum(axis=1) if present_anchor else 0
    )
    folder_pivot["_total_all"] = (
        folder_pivot["_total_global"]
        + folder_pivot["_total_context"]
        + folder_pivot["_total_anchor"]
    )

    sum_cols = [c for c in folder_pivot.columns if c.startswith("_")]
    kw_cols = sorted([c for c in folder_pivot.columns if not c.startswith("_")])
    folder_pivot = folder_pivot[sum_cols + kw_cols]

    folder_pivot = folder_pivot.reset_index()
    folder_pivot = folder_pivot.merge(doc_counts, on=["category", "folder"], how="left")

    return folder_pivot

In [ ]:
def build_summary_table(
    df_raw: pd.DataFrame,
    df_folder: pd.DataFrame,
) -> pd.DataFrame:
    """
    Baut eine Zusammenfassungstabelle auf Ordnerebene mit:
    - Anchor-Hits (total + pro Keyword)
    - Kontext-Keywords (Counts)
    - Kommagetrennte Liste explizit genannter Tools mit Dokumentanteil
    """
    global_names = GLOBAL_KEYWORDS.keys()
    context_names = CONTEXT_KEYWORDS.keys()
    anchor_names = ANCHOR_PATTERNS.keys()

    rows = []

    for _, folder_row in df_folder.iterrows():
        category = folder_row["category"]
        folder = folder_row["folder"]
        n_documents = int(folder_row.get("_n_documents", 0))

        row = {
            "Kategorie": category,
            "Ordner": folder,
            "Dokumente": n_documents,
        }

        # --- Anchor-Hits ---
        anchor_total = 0
        for name in sorted(anchor_names):
            count = folder_row.get(name, 0)
            row[f"Anker: {name}"] = int(count)
            anchor_total += int(count)
        row["Anker: GESAMT"] = anchor_total

        # --- Kontext-Keywords ---
        for name in sorted(context_names):
            count = folder_row.get(name, 0)
            row[f"Kontext: {name}"] = int(count)

        # --- Explizit genannte Tools mit Dokumentanteil ---
        mask = (
            (df_raw["category"] == category)
            & (df_raw["folder"] == folder)
            & (df_raw["search_type"] == "global")
        )
        folder_global_hits = df_raw.loc[mask]

        # Pro Tool: in wie vielen *unterschiedlichen* Dokumenten kommt es vor?
        tools_by_doc = (
            folder_global_hits.groupby("keyword_name")["filename"]
            .nunique()
            .reset_index(name="doc_count")
        )

        tool_list = []
        for _, tool_row in tools_by_doc.iterrows():
            name = tool_row["keyword_name"]
            doc_count = int(tool_row["doc_count"])

            if n_documents > 0:
                pct = doc_count / n_documents * 100
                tool_list.append(f"{name} ({pct:.0f}%)")
            else:
                tool_list.append(f"{name} ({doc_count})")

        row["Explizite PT"] = ", ".join(sorted(tool_list)) if tool_list else "—"

        # Auch als Zahlen
        row["Explizite PT: Anzahl unique"] = len(tool_list)
        row["Explizite PT: Anzahl total"] = (
            int(folder_global_hits.groupby("keyword_name").size().sum())
            if not folder_global_hits.empty
            else 0
        )

        rows.append(row)

    result = pd.DataFrame(rows)

    # --- Kontext-Spalten ohne Hits entfernen ---
    context_cols_all = [c for c in result.columns if c.startswith("Kontext:")]
    context_cols_with_hits = [c for c in context_cols_all if result[c].sum() > 0]
    context_cols_empty = set(context_cols_all) - set(context_cols_with_hits)
    result = result.drop(columns=context_cols_empty)

    # Spaltenreihenfolge ordnen
    meta_cols = ["Kategorie", "Ordner", "Dokumente"]
    anchor_cols = ["Anker: GESAMT"] + sorted(
        [c for c in result.columns if c.startswith("Anker:") and c != "Anker: GESAMT"]
    )
    context_cols = sorted([c for c in result.columns if c.startswith("Kontext:")])
    tool_cols = [
        "Explizite PT: Anzahl unique",
        "Explizite PT: Anzahl total",
        "Explizite PT",
    ]

    result = result[meta_cols + anchor_cols + context_cols + tool_cols]
    return result


def build_tool_detail_table(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Zusätzliche Detailtabelle: Welches Tool in welchem Ordner,
    mit allen Fundstellen.
    """
    global_names = GLOBAL_KEYWORDS.keys()

    mask = df_raw["search_type"] == "global"
    global_hits = df_raw.loc[mask].copy()

    if global_hits.empty:
        return pd.DataFrame()

    # Pivot: Ordner × Tool
    pivot = global_hits.pivot_table(
        index=["category", "folder"],
        columns="keyword_name",
        values="matched_text",
        aggfunc="count",
        fill_value=0,
    )

    # Nur Tools die mindestens 1x vorkommen
    pivot = pivot.loc[:, (pivot > 0).any(axis=0)]

    # Sortieren: häufigste Tools zuerst
    col_order = pivot.sum().sort_values(ascending=False).index
    pivot = pivot[col_order]

    return pivot.reset_index()

## Actual Analysis

### Page Based


In [ ]:
df_raw = analyze_corpus()
df_raw

In [ ]:
df_raw.to_excel("keyword_results/pages/results_raw.xlsx", index=False)
print("→ keyword_results/pages/results_raw.xlsx gespeichert")

In [ ]:
df_doc = aggregate_by_document(df_raw)
df_doc.to_excel("keyword_results/pages/results_by_document.xlsx", index=False)
print(f"→ keyword_results/pages/results_by_document.xlsx ({len(df_doc)} Dokumente)")

df_doc

In [ ]:
# Ordnerebene
df_folder = aggregate_by_folder(df_raw)
df_folder.to_excel("keyword_results/pages/results_by_folder.xlsx", index=False)
print(f"→ keyword_results/pages/results_by_folder.xlsx ({len(df_folder)} Ordner)")

df_folder

In [ ]:
# Kurze Zusammenfassung
print("\n=== Zusammenfassung ===")
print(f"Treffer gesamt:  {len(df_raw)}")
print(f"  davon global:  {len(df_raw[df_raw.search_type == 'global'])}")
print(f"  davon context: {len(df_raw[df_raw.search_type == 'context'])}")
print(f"  davon anchor:  {len(df_raw[df_raw.search_type == 'anchor'])}")
print(f"\nTreffer mit Fuzzy-Fehlern (>0):")
fuzzy_hits = df_raw[df_raw.total_errors > 0]
fuzzy_hits[["keyword_name", "matched_text", "total_errors", "context"]]

In [ ]:
df_tools = build_tool_detail_table(df_raw)
df_tools.to_excel("keyword_results/pages/tool_detail_table.xlsx", index=False)
print(f"\n→ keyword_results/pages/tool_detail_table.xlsx ({len(df_tools)} Zeilen)")
df_tools

In [ ]:
# Haupttabelle
df_summary = build_summary_table(df_raw, df_folder)
df_summary.to_excel("keyword_results/pages/summary_table.xlsx", index=False)
print(f"→ keyword_results/pages/summary_table.xlsx ({len(df_summary)} Zeilen)")
print()
df_summary

### Character based


In [ ]:
WINDOW_MODE = WindowMode.CHARS
WINDOW_SIZE = 50

In [ ]:
df_raw_char = analyze_corpus()
df_raw_char

In [ ]:
df_raw_char.to_excel("keyword_results/characters/results_raw.xlsx", index=False)
print("→ keyword_results/characters/results_raw.xlsx gespeichert")

In [ ]:
df_folder_char = aggregate_by_folder(df_raw_char)
df_folder_char.to_excel(
    "keyword_results/characters/results_by_folder.xlsx", index=False
)
print(f"→ keyword_results/characters/results_by_folder.xlsx ({len(df_folder)} Ordner)")

df_folder

In [ ]:
print("\n=== Zusammenfassung ===")
print(f"Treffer gesamt:  {len(df_raw_char)}")
print(f"  davon global:  {len(df_raw_char[df_raw_char.search_type == 'global'])}")
print(f"  davon context: {len(df_raw_char[df_raw_char.search_type == 'context'])}")
print(f"  davon anchor:  {len(df_raw_char[df_raw_char.search_type == 'anchor'])}")
print(f"\nTreffer mit Fuzzy-Fehlern (>0):")
fuzzy_hits = df_raw_char[df_raw_char.total_errors > 0]
fuzzy_hits[["keyword_name", "matched_text", "total_errors", "context"]]

In [ ]:
df_tools_char = build_tool_detail_table(df_raw_char)
df_tools_char.to_excel("keyword_results/characters/tool_detail_table.xlsx", index=False)
print(
    f"\n→ keyword_results/characters/tool_detail_table.xlsx ({len(df_tools_char)} Zeilen)"
)
df_tools_char

In [ ]:
# Haupttabelle
df_summary = build_summary_table(df_raw_char, df_folder_char)
df_summary.to_excel("keyword_results/characters/summary_table.xlsx", index=False)
print(f"→ keyword_results/characters/summary_table.xlsx ({len(df_summary)} Zeilen)")
df_summary

## Sensitivity Analysis


In [ ]:
@dataclass(frozen=True)
class AnalysisConfig:
    name: str
    window_mode: WindowMode
    window_size: int

    def __str__(self):
        return self.name


BASELINE = AnalysisConfig("chars_±50", WindowMode.CHARS, 50)

SENSITIVITY_CONFIGS = [
    BASELINE,
    # Seitenbasiert
    AnalysisConfig("pages_±0", WindowMode.PAGES, 0),
    AnalysisConfig("pages_±1", WindowMode.PAGES, 1),
    AnalysisConfig("pages_±2", WindowMode.PAGES, 2),
    AnalysisConfig("pages_±3", WindowMode.PAGES, 3),
    AnalysisConfig("pages_±4", WindowMode.PAGES, 4),
    # Zeichenbasiert
    AnalysisConfig("chars_±250", WindowMode.CHARS, 250),
    AnalysisConfig("chars_±500", WindowMode.CHARS, 500),
    AnalysisConfig("chars_±1000", WindowMode.CHARS, 1000),
    AnalysisConfig("chars_±2000", WindowMode.CHARS, 2000),
    AnalysisConfig("chars_±3000", WindowMode.CHARS, 3000),
    AnalysisConfig("chars_±5000", WindowMode.CHARS, 5000),
    # Extremfälle
    AnalysisConfig("entire_document", WindowMode.FULL, 999999),
]

In [ ]:
def run_single_config(config: AnalysisConfig) -> pd.DataFrame:
    """Führt die Analyse mit einer bestimmten Fensterkonfiguration durch."""
    all_results = []
    txt_files = list(SOURCE_DIR.rglob("*.txt"))

    for filepath in sorted(txt_files):
        text = filepath.read_text(encoding="utf-8")
        pages = parse_pages(text)
        full_text = "\n".join(p.text for p in pages)

        results = []

        # 1. Global (unverändert über Szenarien)
        for name, pattern in GLOBAL_KEYWORDS.items():
            for page in pages:
                for match in search_keyword(
                    text=page.text,
                    name=name,
                    pattern=pattern,
                    page_number=page.number,
                    search_type="global",
                ):
                    results.append(match.to_dict(filepath))

        # 2. Anker finden (unverändert über Szenarien)
        anchors = find_anchors_with_positions(pages)
        for anchor in anchors:
            results.append(anchor.match.to_dict(filepath))

        # 3. Kontextfenster (VARIIERT über Szenarien)
        windows = build_context_windows(
            pages=pages,
            full_text=full_text,
            anchors=anchors,
            mode=config.window_mode,
            window_size=config.window_size,
        )
        for window_text, page_numbers in windows:
            for name, pattern in CONTEXT_KEYWORDS.items():
                for match in search_keyword(
                    text=window_text,
                    name=name,
                    pattern=pattern,
                    search_type="context",
                ):
                    match.page = page_numbers[0]
                    results.append(match.to_dict(filepath))

        all_results.extend(results)

    df = pd.DataFrame(all_results)
    df["config"] = config.name
    return df


def run_sensitivity_analysis(analyses_params: list[AnalysisConfig]) -> pd.DataFrame:
    """Runs the given analyses.

    Returns:
        pd.DataFrame: Result of sensetivity analysis
    """
    all_dfs = []
    for i, config in enumerate(analyses_params):
        print(f"[{i+1}/{len(analyses_params)}] {config.name}")
        df = run_single_config(config)
        all_dfs.append(df)
        print(f"  → {len(df[df.search_type == 'context'])} Kontext-Treffer")
    return pd.concat(all_dfs, ignore_index=True)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams["figure.dpi"] = 150


def build_comparison_table(
    df_all: pd.DataFrame, baseline_name: str | None = None
) -> pd.DataFrame:
    """Vergleichstabelle: Kontext-Treffer pro Szenario vs. Baseline."""

    configs = df_all["config"].unique()

    if baseline_name is None:
        # Erste Config als Baseline nehmen
        baseline_name = configs[0]
        print(f"Auto-detected baseline: '{baseline_name}'")

    print(configs)

    rows = []
    for config_name in configs:
        df_ctx = df_all[
            (df_all["config"] == config_name) & (df_all["search_type"] == "context")
        ]

        n_hits = len(df_ctx)
        n_unique_kw = df_ctx["keyword_name"].nunique()
        n_folders = df_ctx["folder"].nunique()
        n_docs = df_ctx["filename"].nunique()

        rows.append(
            {
                "Szenario": config_name,
                "Kontext-Treffer": n_hits,
                "Unique Keywords": n_unique_kw,
                "Ordner mit Hits": n_folders,
                "Dokumente mit Hits": n_docs,
            }
        )

    result = pd.DataFrame(rows)

    # Abweichung zur Baseline
    baseline_hits = result.loc[
        result["Szenario"] == baseline_name, "Kontext-Treffer"
    ].iloc[0]

    if baseline_hits > 0:
        result["Δ Treffer (%)"] = (
            (result["Kontext-Treffer"] - baseline_hits) / baseline_hits * 100
        ).round(1)
    else:
        result["Δ Treffer (%)"] = 0.0

    # Sortieren: Pages zuerst, dann Chars, nach Größe
    return result


def build_keyword_stability(df_all: pd.DataFrame) -> pd.DataFrame:
    """
    Pro Kontext-Keyword: in wie vielen Szenarien wird es gefunden?
    Zeigt welche Keywords stabil sind und welche vom Fenster abhängen.
    """
    ctx_hits = df_all[df_all["search_type"] == "context"]
    n_configs = df_all["config"].nunique()

    # Pro keyword × config: gefunden ja/nein
    presence = (
        ctx_hits.groupby(["keyword_name", "config"]).size().reset_index(name="count")
    )
    presence["found"] = 1

    pivot = presence.pivot_table(
        index="keyword_name",
        columns="config",
        values="found",
        fill_value=0,
    )

    pivot["Szenarien mit Hit"] = pivot.sum(axis=1).astype(int)
    pivot["Stabilität (%)"] = (pivot["Szenarien mit Hit"] / n_configs * 100).round(1)

    return pivot.sort_values("Stabilität (%)", ascending=False).reset_index()


def build_folder_stability(df_all: pd.DataFrame) -> pd.DataFrame:
    """
    Pro Ordner x Keyword: in wie vielen Szenarien wird es gefunden?
    Identifiziert fenstergröße-sensitive Ergebnisse.
    """
    ctx_hits = df_all[df_all["search_type"] == "context"]
    n_configs = df_all["config"].nunique()

    presence = (
        ctx_hits.groupby(["folder", "keyword_name", "config"])
        .size()
        .reset_index(name="count")
    )
    presence["found"] = 1

    pivot = presence.pivot_table(
        index=["folder", "keyword_name"],
        columns="config",
        values="found",
        fill_value=0,
    )

    pivot["Szenarien mit Hit"] = pivot.sum(axis=1).astype(int)
    pivot["Stabilität (%)"] = (pivot["Szenarien mit Hit"] / n_configs * 100).round(1)

    return pivot.sort_values(
        ["folder", "Stabilität (%)"], ascending=[True, False]
    ).reset_index()


def build_hit_count_by_window(df_all: pd.DataFrame) -> pd.DataFrame:
    """
    Treffer pro Keyword x Fenstergröße.
    Zeigt den Sättigungsverlauf.
    """
    ctx_hits = df_all[df_all["search_type"] == "context"]

    counts = (
        ctx_hits.groupby(["config", "keyword_name"]).size().reset_index(name="count")
    )

    pivot = counts.pivot_table(
        index="keyword_name",
        columns="config",
        values="count",
        fill_value=0,
    )

    return pivot.reset_index()


# ============================================================
# Visualisierungen
# ============================================================


def plot_treffer_by_window(df_all: pd.DataFrame):
    """Liniendiagramm: Kontext-Treffer als Funktion der Fenstergröße."""
    ctx_hits = df_all[df_all["search_type"] == "context"]

    counts = ctx_hits.groupby("config").size().reset_index(name="count")

    # Fenstergröße extrahieren für Sortierung
    def extract_size(name):
        parts = name.split("±")
        if len(parts) == 2:
            try:
                return int(parts[1])
            except ValueError:
                return 0
        return 0

    counts["size"] = counts["config"].apply(extract_size)
    counts["mode"] = counts["config"].apply(
        lambda x: (
            "Seiten" if "pages" in x else "Zeichen" if "chars" in x else "Sonstige"
        )
    )

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Seitenbasiert
    pages = counts[counts["mode"] == "Seiten"].sort_values("size")
    ax1.plot(pages["size"], pages["count"], "o-", color="steelblue", linewidth=2)
    ax1.set_xlabel("Fenstergröße (±Seiten)")
    ax1.set_ylabel("Kontext-Treffer")
    ax1.set_title("Seitenbasiertes Fenster")
    ax1.grid(True, alpha=0.3)

    # Zeichenbasiert
    chars = counts[counts["mode"] == "Zeichen"].sort_values("size")
    ax2.plot(chars["size"], chars["count"], "s-", color="coral", linewidth=2)
    ax2.set_xlabel("Fenstergröße (±Zeichen)")
    ax2.set_ylabel("Kontext-Treffer")
    ax2.set_title("Zeichenbasiertes Fenster")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(
        DEST_DIR / SENSITIVITY_DIR / "sensitivity_window_size.pdf", bbox_inches="tight"
    )
    plt.savefig(
        DEST_DIR / SENSITIVITY_DIR / "sensitivity_window_size.png", bbox_inches="tight"
    )
    plt.show()


def plot_keyword_heatmap(df_all: pd.DataFrame, log_scale: bool = True, sort_columns: str = "window_size",):
    """Heatmap: Keyword x Szenario → Trefferanzahl."""
    import matplotlib.pyplot as plt
    from matplotlib.colors import LogNorm, Normalize
    import numpy as np

    ctx_hits = df_all[df_all["search_type"] == "context"]

    pivot = ctx_hits.pivot_table(
        index="keyword_name",
        columns="config",
        values="matched_text",
        aggfunc="count",
        fill_value=0,
    )

    # Nur Keywords mit Variation
    variation = pivot.max(axis=1) - pivot.min(axis=1)
    pivot = pivot[variation > 0]

    if pivot.empty:
        print("Alle Keywords sind perfekt stabil - keine Heatmap nötig.")
        return

    # --- Spalten sortieren ---
    def extract_sort_key(name):
        """Erzeugt Sortier-Tupel: (Modus-Rang, Größe)."""
        if "entire" in name:
            return (2, 999_999)
        if "only_anchor" in name:
            return (0, -1)

        parts = name.split("±")
        if len(parts) == 2:
            try:
                size = int(parts[1])
            except ValueError:
                size = 0
        else:
            size = 0

        if "pages" in name:
            mode_rank = 0
        elif "chars" in name:
            mode_rank = 1
        else:
            mode_rank = 2

        return (mode_rank, size)

    if sort_columns == "window_size":
        sorted_cols = sorted(pivot.columns, key=extract_sort_key)
    elif sort_columns == "alphabetical":
        sorted_cols = sorted(pivot.columns)
    elif sort_columns == "total_hits":
        sorted_cols = pivot.sum().sort_values(ascending=False).index.tolist()

    pivot = pivot[sorted_cols]

    # Zeilen nach Gesamttrefferanzahl sortieren
    pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]

    # --- Plot ---
    fig, ax = plt.subplots(
        figsize=(max(12, len(pivot.columns) * 0.9),
                 max(6, len(pivot) * 0.45))
    )

    plot_data = pivot.values.astype(float)

    if log_scale:
        # 0-Werte auf NaN setzen damit LogNorm funktioniert
        plot_data_masked = np.where(plot_data > 0, plot_data, np.nan)
        vmin = max(1, np.nanmin(plot_data_masked)) if np.any(~np.isnan(plot_data_masked)) else 1
        vmax = np.nanmax(plot_data_masked) if np.any(~np.isnan(plot_data_masked)) else 1
        norm = LogNorm(vmin=vmin, vmax=vmax)
        # NaN als eigene Farbe (weiß)
        cmap = plt.cm.YlOrRd.copy()
        cmap.set_bad(color="white")
        im = ax.imshow(plot_data_masked, aspect="auto", cmap=cmap, norm=norm)
    else:
        norm = Normalize(vmin=0, vmax=plot_data.max())
        im = ax.imshow(plot_data, aspect="auto", cmap="YlOrRd", norm=norm)

        # Achsenbeschriftung
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=8)

    # Werte in Zellen schreiben
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = int(pivot.iloc[i, j])
            if val > 0:
                # Textfarbe: weiß auf dunklem Hintergrund
                text_color = "white" if val > (vmax * 0.4 if log_scale else plot_data.max() * 0.6) else "black"
                ax.text(j, i, str(val), ha="center", va="center",
                        fontsize=7, color=text_color)

    cbar = plt.colorbar(im, ax=ax, label="Treffer")
    if log_scale:
        cbar.set_label("Treffer (log. Skala)")

    ax.set_title("Kontext-Keywords: Sensitivität gegenüber Fenstergröße")

    plt.tight_layout()
    plt.savefig(
        DEST_DIR / SENSITIVITY_DIR / "sensitivity_heatmap.pdf", bbox_inches="tight"
    )
    plt.savefig(
        DEST_DIR / SENSITIVITY_DIR / "sensitivity_heatmap.png", bbox_inches="tight"
    )
    plt.show()

In [ ]:
df_all = run_sensitivity_analysis(SENSITIVITY_CONFIGS)
df_all.to_excel("keyword_results/sensitivity/sensitivity_raw.xlsx", index=False)
df_all

In [ ]:
df_compare = build_comparison_table(df_all)
df_compare.to_excel(
    DEST_DIR / SENSITIVITY_DIR / "sensitivity_comparison.xlsx", index=False
)
print(f"→ {DEST_DIR / SENSITIVITY_DIR}/sensitivity_comparison.xlsx")
df_compare

In [ ]:
df_folder_stab = build_folder_stability(df_all)
df_folder_stab.to_excel(
    DEST_DIR / SENSITIVITY_DIR / "sensitivity_folder_stability.xlsx", index=False
)
print(f"→ {DEST_DIR / SENSITIVITY_DIR}/sensitivity_folder_stability.xlsx")
df_folder_stab

In [ ]:
df_saturation = build_hit_count_by_window(df_all)
df_saturation.to_excel(
    DEST_DIR / SENSITIVITY_DIR / "sensitivity_saturation.xlsx", index=False
)
print(f"→ {DEST_DIR / SENSITIVITY_DIR}/sensitivity_saturation.xlsx")
df_saturation

In [ ]:
plot_keyword_heatmap(df_all)

In [ ]:
plot_treffer_by_window(df_all)

In [ ]:
def find_window_sensitive_docs(
    df_all: pd.DataFrame,
    keyword: str,
    config_large: str = "entire_document",
    config_small: str = "chars_±50",
) -> pd.DataFrame:
    ctx = df_all[
        (df_all["search_type"] == "context")
        & (df_all["keyword_name"] == keyword)
    ]

    large = (
        ctx[ctx["config"] == config_large]
        .groupby(["folder", "filename"])
        .size()
        .reset_index(name="hits_large")
    )

    small = (
        ctx[ctx["config"] == config_small]
        .groupby(["folder", "filename"])
        .size()
        .reset_index(name="hits_small")
    )

    result = large.merge(small, on=["folder", "filename"], how="left")
    result["hits_small"] = result["hits_small"].fillna(0).astype(int)
    result["diff"] = result["hits_large"] - result["hits_small"]

    return result.sort_values("diff", ascending=False)

In [ ]:
find_window_sensitive_docs(df_all, "ctx_blockbasiert")